# Concurrency and Performance

Notebook for **Chapter 11** of the *python* study document. Run the cells top to bottom. Read the full explanation on the site: https://dadipavan.github.io/Learning/study/python/ch11.html

Tip: change a number, predict the output, then run the cell again.

## 11.1 Measure First: timeit, cProfile and tracemalloc

In [ ]:
import timeit
N=1_000_000
t1=timeit.timeit("s=0\nfor i in range(N): s+=i*i", globals=globals(), number=3)/3
t2=timeit.timeit("sum([i*i for i in range(N)])", globals=globals(), number=3)/3
t3=timeit.timeit("sum(i*i for i in range(N))", globals=globals(), number=3)/3
t4=timeit.timeit("sum(map(int.__mul__, r, r))", setup="r=range(N)", globals=globals(), number=3)/3
for n,t in [("for loop",t1),("list comp + sum",t2),("generator + sum",t3),("map(int.__mul__)",t4)]:
    print(f"{n:<20}{t*1000:8.1f} ms")

for loop                27.2 ms
list comp + sum         26.3 ms
generator + sum         28.8 ms
map(int.__mul__)        45.7 ms


In [ ]:
import timeit
print(timeit.timeit("99999 in L", setup="L=list(range(100000))", number=200), "s list")
print(timeit.timeit("99999 in S", setup="S=set(range(100000))", number=200), "s set")

0.08791520900012983 s list
4.417000127432402e-06 s set


In [ ]:
import cProfile, pstats, io
def slow_sum(n):
    return sum([i*i for i in range(n)])
def fib(n):
    return n if n < 2 else fib(n-1)+fib(n-2)
def main():
    slow_sum(300_000)
    fib(22)
pr = cProfile.Profile(); pr.enable(); main(); pr.disable()
s = io.StringIO()
pstats.Stats(pr, stream=s).sort_stats("cumulative").print_stats(5)
print(s.getvalue())

         57317 function calls (5 primitive calls) in 0.014 seconds

   Ordered by: cumulative time

   ncalls  tottime  percall  cumtime  percall filename:lineno(function)
        1    0.000    0.000    0.014    0.014 /private/tmp/w11/b.py:6(main)
        1    0.006    0.006    0.008    0.008 /private/tmp/w11/b.py:2(slow_sum)
  57313/1    0.006    0.000    0.006    0.006 /private/tmp/w11/b.py:4(fib)
        1    0.002    0.000    0.002    0.002 {built-in method builtins.sum}
        1    0.000    0.000    0.000    0.000 {method 'disable' of '_lsprof.Profiler' objects}


In [ ]:
import tracemalloc, sys
tracemalloc.start()
data = [i for i in range(1_000_000)]
cur, peak = tracemalloc.get_traced_memory()
print(f"list of 1M ints: current {cur/1e6:.1f} MB")
tracemalloc.reset_peak()
gen = (i for i in range(1_000_000))
total = sum(gen)
cur2, peak2 = tracemalloc.get_traced_memory()
print(f"generator sum peak extra: {(peak2-cur)/1e6:.4f} MB")
print(sys.getsizeof(data), sys.getsizeof(gen))
snap = tracemalloc.take_snapshot()
for st in snap.statistics("lineno")[:1]: print(st)

list of 1M ints: current 40.4 MB
generator sum peak extra: 0.0006 MB
8448728 200
/private/tmp/w11/d.py:3: size=38.6 MiB, count=999731, average=40 B


## 11.2 The GIL: Threads, Processes and Who Wins

In [ ]:
import time
from concurrent.futures import ThreadPoolExecutor, ProcessPoolExecutor

def io_task(_):
    time.sleep(0.5)
def cpu_task(n):
    s = 0
    for i in range(n):
        s += i*i
    return s

def timed(label, f):
    t=time.perf_counter(); f(); print(f"{label:<28}{time.perf_counter()-t:6.2f} s")

N=30_000_000
if __name__ == "__main__":
    timed("IO x8 sequential", lambda: [io_task(0) for _ in range(8)])
    def tp():
        with ThreadPoolExecutor(8) as ex: list(ex.map(io_task, range(8)))
    timed("IO x8 threads(8)", tp)
    timed("CPU x4 sequential", lambda: [cpu_task(N) for _ in range(4)])
    def tc():
        with ThreadPoolExecutor(4) as ex: list(ex.map(cpu_task,[N]*4))
    timed("CPU x4 threads(4)", tc)
    def pc():
        with ProcessPoolExecutor(4) as ex: list(ex.map(cpu_task,[N]*4))
    timed("CPU x4 processes(4)", pc)

In [ ]:
# This cell is meant to fail: read the error message, that is the lesson.
from concurrent.futures import ProcessPoolExecutor
with ProcessPoolExecutor(2) as ex:
    print(list(ex.map(lambda x: x*2, [1,2])))

PicklingError - Can't pickle <function <lambda> at 0x1039e6770>: it's not found as __m


In [ ]:
import threading, time
counter = 0
lock = threading.Lock()
def unsafe(n):
    global counter
    for _ in range(n):
        v = counter
        time.sleep(0)       # gives other threads a chance to run mid-update
        counter = v + 1
def safe(n):
    global counter
    for _ in range(n):
        with lock:
            v = counter
            time.sleep(0)
            counter = v + 1
def run(fn):
    global counter
    counter = 0
    ts=[threading.Thread(target=fn,args=(2000,)) for _ in range(4)]
    [t.start() for t in ts]; [t.join() for t in ts]
    return counter
print("unsafe:", run(unsafe), "expected 8000")
print("safe:  ", run(safe), "expected 8000")

unsafe: 2015 expected 8000
safe:   8000 expected 8000


## 11.3 concurrent.futures and asyncio

In [ ]:
from concurrent.futures import ThreadPoolExecutor, as_completed
import time, random
def job(i):
    time.sleep(random.random()*0.3)
    if i == 3: raise ValueError("bad item 3")
    return i*10
with ThreadPoolExecutor(4) as ex:
    futs = {ex.submit(job, i): i for i in range(6)}
    for f in as_completed(futs):
        i = futs[f]
        try: print(i, "->", f.result())
        except ValueError as e: print(i, "failed:", e)

In [ ]:
import asyncio, time
async def fetch(i, sem):
    async with sem:
        await asyncio.sleep(0.2)
        return i*i
async def main():
    sem = asyncio.Semaphore(5)
    t=time.perf_counter()
    res = await asyncio.gather(*(fetch(i, sem) for i in range(20)))
    print(res[:5], f"{time.perf_counter()-t:.2f}s")
    try:
        async with asyncio.timeout(0.1):
            await asyncio.sleep(1)
    except TimeoutError:
        print("timed out")
asyncio.run(main())

[0, 1, 4, 9, 16] 0.80s
timed out


In [ ]:
import asyncio, time
async def bad(i):
    time.sleep(0.2)          # blocks the whole event loop
    return i
async def good(i):
    await asyncio.sleep(0.2)
    return i
async def main():
    t=time.perf_counter(); await asyncio.gather(*(bad(i) for i in range(5))); print(f"time.sleep in async : {time.perf_counter()-t:.2f}s")
    t=time.perf_counter(); await asyncio.gather(*(good(i) for i in range(5))); print(f"await asyncio.sleep : {time.perf_counter()-t:.2f}s")
    t=time.perf_counter(); await asyncio.gather(*(asyncio.to_thread(time.sleep,0.2) for i in range(5))); print(f"to_thread(time.sleep): {time.perf_counter()-t:.2f}s")
asyncio.run(main())

time.sleep in async : 1.02s
await asyncio.sleep : 0.20s
to_thread(time.sleep): 0.21s


## 11.4 Vectorization, Algorithms and Native Speed

In [ ]:
import time, statistics
n=1_000_000
xs=list(range(n))
t=time.perf_counter(); ys=[x*2.5+1 for x in xs]; print(f"list comp {time.perf_counter()-t:.3f}s")
t=time.perf_counter()
ys=[0.0]*n
for i in range(n): ys[i]=xs[i]*2.5+1
print(f"index loop {time.perf_counter()-t:.3f}s")
t=time.perf_counter(); statistics.mean(xs); print(f"statistics.mean {time.perf_counter()-t:.3f}s")
t=time.perf_counter(); sum(xs)/len(xs); print(f"sum/len {time.perf_counter()-t:.4f}s")

list comp 0.026s
index loop 0.056s
statistics.mean 0.091s
sum/len 0.0032s


In [ ]:
# not run here: needs numpy and pandas
import numpy as np
x = np.arange(1_000_000)
y = x * 2.5 + 1            # whole array, no Python loop
print(y.mean())

import pandas as pd
df = pd.DataFrame({"qty": [2, 5, 1], "price": [100.0, 40.0, 250.0]})
df["total"] = df["qty"] * df["price"]      # vectorized column maths
# slow pattern to avoid:
# df["total"] = df.apply(lambda r: r["qty"] * r["price"], axis=1)

In [ ]:
import time
from multiprocessing import Pool
def count_primes(rng):
    lo, hi = rng
    c = 0
    for n in range(lo, hi):
        if n < 2: continue
        for d in range(2, int(n**0.5)+1):
            if n % d == 0: break
        else: c += 1
    return c
if __name__ == "__main__":
    chunks = [(i*250_000, (i+1)*250_000) for i in range(8)]
    t=time.perf_counter(); r1=sum(map(count_primes, chunks)); t1=time.perf_counter()-t
    t=time.perf_counter()
    with Pool(4) as p: r2=sum(p.map(count_primes, chunks))
    t2=time.perf_counter()-t
    print(r1, r2, f"serial {t1:.2f}s  pool(4) {t2:.2f}s  speedup {t1/t2:.1f}x")

148933 148933 serial 3.75s  pool(4) 1.49s  speedup 2.5x


## 11.5 Queues, Back-Pressure and Little's Law

In [ ]:
import queue, threading, time
q = queue.Queue(maxsize=3)
results = []
def producer():
    for i in range(8):
        q.put(i)               # blocks when queue is full (back-pressure)
    for _ in range(2): q.put(None)
def consumer(name):
    while (item := q.get()) is not None:
        time.sleep(0.01)
        results.append((name, item))
ts=[threading.Thread(target=producer)]+[threading.Thread(target=consumer,args=(f"c{i}",)) for i in range(2)]
[t.start() for t in ts]; [t.join() for t in ts]
print(sorted(i for _,i in results))
print(len(results), "items processed")

[0, 1, 2, 3, 4, 5, 6, 7]
8 items processed


In [ ]:
import threading, time
LAT = 0.05          # each request takes 50 ms
DURATION = 2.0
def run(workers):
    done = 0
    lock = threading.Lock()
    end = time.perf_counter() + DURATION
    def w():
        nonlocal done
        while time.perf_counter() < end:
            time.sleep(LAT)
            with lock: done += 1
    ts=[threading.Thread(target=w) for _ in range(workers)]
    t0=time.perf_counter()
    [t.start() for t in ts]; [t.join() for t in ts]
    el=time.perf_counter()-t0
    return done/el
for c in (1,5,10,20):
    print(f"concurrency {c:>2}: measured {run(c):7.1f} req/s   predicted {c/LAT:7.1f} req/s")

concurrency  1: measured    18.6 req/s   predicted    20.0 req/s
concurrency  5: measured    92.6 req/s   predicted   100.0 req/s
concurrency 10: measured   185.3 req/s   predicted   200.0 req/s
concurrency 20: measured   370.6 req/s   predicted   400.0 req/s


## 11.6 Modern asyncio: TaskGroup, Timeouts, Limits, Queues and Cancellation

Section 11.3 introduced `async` and `await`. Real programs also need to stop work that takes too long, cap how much runs at once, pass work between tasks, and clean up after cancellation. Python 3.11 and newer give you clean tools for each. All blocks below were run.

In [ ]:
import asyncio

async def fetch(name, delay, fail=False):
    await asyncio.sleep(delay)
    if fail:
        raise ValueError(name)
    return name

async def main():
    async with asyncio.TaskGroup() as tg:
        t1 = tg.create_task(fetch("a", 0.02))
        t2 = tg.create_task(fetch("b", 0.01))
    print(t1.result(), t2.result())
    try:
        async with asyncio.TaskGroup() as tg:
            tg.create_task(fetch("ok", 0.5))
            tg.create_task(fetch("bad", 0.01, fail=True))
    except* ValueError as eg:
        print("caught", [str(e) for e in eg.exceptions])
    try:
        async with asyncio.timeout(0.05):
            await asyncio.sleep(1)
    except TimeoutError:
        print("timed out")
asyncio.run(main())

a b
caught ['bad']
timed out


In [ ]:
import asyncio

async def worker(i, sem, state):
    async with sem:
        state["now"] += 1
        state["peak"] = max(state["peak"], state["now"])
        await asyncio.sleep(0.01)
        state["now"] -= 1
    return i

async def producer(q, n):
    for i in range(n):
        await q.put(i)
    await q.put(None)

async def consumer(q, out):
    while (item := await q.get()) is not None:
        out.append(item * item)

async def main():
    sem = asyncio.Semaphore(3)
    state = {"now": 0, "peak": 0}
    res = await asyncio.gather(*(worker(i, sem, state) for i in range(10)))
    print(res, "peak concurrency", state["peak"])
    q = asyncio.Queue(maxsize=2)
    out = []
    await asyncio.gather(producer(q, 6), consumer(q, out))
    print(out)
    r = await asyncio.to_thread(sum, range(1000))
    print("thread result", r)
asyncio.run(main())

[0, 1, 2, 3, 4, 5, 6, 7, 8, 9] peak concurrency 3
[0, 1, 4, 9, 16, 25]
thread result 499500


In [ ]:
import asyncio

async def forever():
    try:
        await asyncio.sleep(10)
    except asyncio.CancelledError:
        print("cleanup ran")
        raise

async def main():
    t = asyncio.create_task(forever())
    await asyncio.sleep(0.01)
    t.cancel()
    try:
        await t
    except asyncio.CancelledError:
        print("task cancelled:", t.cancelled())
asyncio.run(main())

cleanup ran
task cancelled: True


## 11.7 Locks, Deadlocks, Memory and Data-Structure Speed

Threads share memory, and that is where the trouble starts. This section shows the lost update, the lock that cures it, a deadlock and its fix, then moves to memory: reference counting, cycles, weak references and the data structure that makes queues cheap.

In [ ]:
import threading

def run(use_lock, n=200_000):
    counter = {"v": 0}
    lock = threading.Lock()
    def work():
        for _ in range(n):
            if use_lock:
                with lock:
                    counter["v"] += 1
            else:
                v = counter["v"]
                counter["v"] = v + 1
    ts = [threading.Thread(target=work) for _ in range(4)]
    for t in ts: t.start()
    for t in ts: t.join()
    return counter["v"]
print(run(True))
print(run(False) <= 800_000)

800000
True


In [ ]:
import threading, time
a, b = threading.Lock(), threading.Lock()
log = []
def t1():
    with a:
        time.sleep(0.05)
        if b.acquire(timeout=0.3):
            b.release(); log.append("t1 got both")
        else:
            log.append("t1 gave up")
def t2():
    with b:
        time.sleep(0.05)
        if a.acquire(timeout=0.3):
            a.release(); log.append("t2 got both")
        else:
            log.append("t2 gave up")
x = [threading.Thread(target=t1), threading.Thread(target=t2)]
for t in x: t.start()
for t in x: t.join()
print(sorted(log))
def ordered(name):
    with a:
        with b:
            log.append(name + " ordered ok")
log.clear()
y = [threading.Thread(target=ordered, args=(n,)) for n in ("t1", "t2")]
for t in y: t.start()
for t in y: t.join()
print(sorted(log))

['t1 gave up', 't2 gave up']
['t1 ordered ok', 't2 ordered ok']


In [ ]:
import gc, weakref

class Node:
    def __init__(self): self.other = None
gc.disable()
a, b = Node(), Node()
a.other, b.other = b, a
r = weakref.ref(a)
del a, b
print("alive after del:", r() is not None)
print("collected objects > 0:", gc.collect() > 0)
print("alive after gc:", r() is not None)
gc.enable()

class Big: pass
cache = weakref.WeakValueDictionary()
obj = Big()
cache["k"] = obj
print(len(cache))
del obj
print(len(cache))

alive after del: True
collected objects > 0: True
alive after gc: False
1
0


In [ ]:
import timeit

n = 50_000
tl = timeit.timeit("l.pop(0)", setup=f"l=list(range({n}))", number=n-1)
td = timeit.timeit("d.popleft()", setup=f"from collections import deque; d=deque(range({n}))", number=n-1)
print("deque faster than list.pop(0) by more than 10x:", tl / td > 10)

deque faster than list.pop(0) by more than 10x: True


In [ ]:
import asyncio

async def fetch(name, delay):
    await asyncio.sleep(delay)
    return name

async def main():
    print(await asyncio.gather(fetch("a", 0.03), fetch("b", 0.01)))
asyncio.run(main())

['a', 'b']


## Chapter recap

## Practice set

Problem 11.P.1 (easy) A job has `L = 10` concurrent workers and each request takes `W = 0.2 s`. What is the throughput?

In [ ]:
import asyncio
async def f():
    return 5
print(f())

In [ ]:
with ThreadPoolExecutor(8) as ex:
    results = list(ex.map(heavy_math, data))

## Prompt pack

---
Done? Do the **Practice set** at the end of the chapter, then clear your Review cards on the site.

https://dadipavan.github.io/Learning/study/python/ch11.html